# 🏇 KEIBA-AI 土日ノート v8

| タイミング | 実行セル | 内容 |
|---|---|---|
| 土曜夜 | 1→2→3→4→5→6→7 | 結果取得→バイアス→日曜予想→Push |
| 日曜夜 | 1→2→3→4 | 結果取得のみ |

In [ ]:
# ── セル1: セットアップ ──
!pip install -q requests beautifulsoup4 lxml pandas

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, sqlite3, requests, pickle, statistics
from datetime import datetime, timezone, timedelta
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_DIR = '/content/drive/MyDrive/keiba_ai'
for d in [BASE_DIR, f'{BASE_DIR}/data', f'{BASE_DIR}/app']:
    os.makedirs(d, exist_ok=True)

JST = timezone(timedelta(hours=9))
jst_now = datetime.now(JST)
print(f'✅ セットアップ完了 ({jst_now.strftime("%Y-%m-%d %H:%M:%S")} JST)')

In [ ]:
# ── セル2: 強制アップデート（毎回実行） ──
import urllib.request, time as _time

BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
SRC_FILES = [
    'src/__init__.py',
    'src/tools/__init__.py',
    'src/tools/tune_weights.py',
    'src/tools/calibrate.py',
    'src/tools/analyze_divergence.py',
    'src/tools/rescrape_history.py',
    'src/tools/bias.py',
    'src/features/__init__.py',
    'src/features/engine.py',
    'src/features/speed_index.py',
    'src/utils/__init__.py',
    'src/utils/config.py',
    'src/utils/db.py',
    'src/utils/model_registry.py',
    'src/utils/jst.py',
    'src/scraper/__init__.py',
    'src/scraper/parser.py',
    'src/scraper/jra_scraper.py',
    'src/scraper/calendar.py',
    'src/models/__init__.py',
    'src/models/calibration.py',
    'src/models/predict.py',
    'src/betting/__init__.py',
    'src/betting/make_bets.py',
    'src/betting/ev_filter.py',
    'src/betting/app_json.py',
]
ts = int(_time.time())
for rel in SRC_FILES:
    dest = f'{BASE_DIR}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    urllib.request.urlretrieve(f'{BASE_URL}/{rel}?t={ts}', dest)
    print(f'✅ {rel}')
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]
sys.path.insert(0, BASE_DIR)
print('🔄 アップデート完了')

In [ ]:
# ── セル3: 設定・初期化（毎回実行） ──
# ▼▼ 賭け金はここで変更 ▼▼
BANKROLL    = 100_000
TOP_N_RACES = 6
FUKU_AMT = 500
TAN_AMT  = 300
WIDE_AMT = 300
REN_AMT  = 300
TAN2_AMT = 300
SAN_AMT  = 300

from src.features.engine import init_engine
from src.utils.db import (init_db, save_race_db, save_bets_db,
                           save_history_db, save_results_db, check_and_update_bets)
from src.betting.make_bets import init_betting, make_bets, log_bet_simulation
from src.betting.ev_filter import ability_first_loose
from src.betting.app_json import to_app_json
from src.scraper.jra_scraper import fetch_races_on_date, fetch_results

HEADERS   = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
JRA_BASE  = 'https://www.jra.go.jp'
HIST_PATH = f'{BASE_DIR}/data/history.db'
BIAS_PATH = f'{BASE_DIR}/data/track_bias_latest.json'

init_db(BASE_DIR)
init_engine(BASE_DIR)
init_betting(BASE_DIR, bankroll=BANKROLL,
             fuku_amt=FUKU_AMT, tan_amt=TAN_AMT, wide_amt=WIDE_AMT,
             ren_amt=REN_AMT, tan2_amt=TAN2_AMT, san_amt=SAN_AMT)

avg_bias = None
if os.path.exists(BIAS_PATH):
    with open(BIAS_PATH) as f:
        avg_bias = json.load(f)
    print(f'  📊 バイアス({avg_bias.get("date","")}): {avg_bias.get("summary","フラット")}')
else:
    print('  📊 バイアス: なし（フラット想定）')

print(f'✅ 初期化完了  複勝¥{FUKU_AMT} 単勝¥{TAN_AMT} ワイド¥{WIDE_AMT}')

In [ ]:
# ── セル4: ★ 結果取得・history.db保存・bet照合 ──
# 土曜夜 → 当日(土)の結果  /  日曜夜 → 当日(日)の結果

jst_now = datetime.now(JST)
target_date = jst_now.strftime('%Y%m%d')
print(f'📅 本日結果取得: {target_date}')

sess = requests.Session()
sess.headers.update(HEADERS)
retry = Retry(total=3, backoff_factor=2, status_forcelist=[500,502,503,504])
sess.mount('https://', HTTPAdapter(max_retries=retry))
sess.get(f'{JRA_BASE}/keiba/thisweek/', timeout=15)

all_results = fetch_results(sess, target_date)
print(f'📋 取得完了: {len(all_results)}レース')

if all_results:
    surf_counts = {}
    for r in all_results:
        s = r.get('surface', '?')
        surf_counts[s] = surf_counts.get(s, 0) + 1
    print(f'   コース内訳: {surf_counts}')

    save_history_db(all_results, BASE_DIR)
    save_results_db(all_results, BASE_DIR)

    conn = sqlite3.connect(f'{BASE_DIR}/data/history.db')
    dt_max = conn.execute('SELECT MAX(date) FROM race_history').fetchone()[0]
    rc = conn.execute('SELECT COUNT(*) FROM race_history').fetchone()[0]
    hc = conn.execute('SELECT COUNT(*) FROM horse_history').fetchone()[0]
    conn.close()
    print(f'   📊 history.db: {rc:,}レース / {hc:,}出走 / 最新: {dt_max}')

chk = check_and_update_bets(all_results, BASE_DIR)
print(f'\n【照合結果】 {chk["hit"]}/{chk["total"]}的中  '
      f'投資¥{chk["invested"]:,} / 回収¥{chk["recovered"]:,}  '
      f'ROI {chk["roi"]:.1f}%')
for d in chk['details']:
    print(d)

In [ ]:
# ── セル5: ★ バイアス分析・JSON保存（土曜夜のみ） ──

from src.tools.bias import analyze_bias, build_avg_bias

bias_by_course = analyze_bias(all_results)
print('【トラックバイアス分析】')
for rc, b in bias_by_course.items():
    print(f'  {rc}: {b["summary"]}  '
          f'内外:{b["inner_outer"]:+.2f} ペース:{b["pace_bias"]:+.2f} '
          f'時計:{b["track_speed"]:+.2f}  ({b["race_count"]}R)')

prev_bias = None
if os.path.exists(BIAS_PATH):
    with open(BIAS_PATH) as f:
        prev_bias = json.load(f)

avg_bias = build_avg_bias(bias_by_course, prev_bias)
print(f'\n【全体バイアス】{avg_bias["summary"]}')
print(f'  内外:{avg_bias["inner_outer"]:+.2f}  ペース:{avg_bias["pace_bias"]:+.2f}  '
      f'時計:{avg_bias["track_speed"]:+.2f}')

with open(BIAS_PATH, 'w', encoding='utf-8') as f:
    json.dump(avg_bias, f, ensure_ascii=False, indent=2)
print(f'✅ バイアス保存: {BIAS_PATH}')

In [ ]:
# ── セル6: ★ 翌日（日曜）レース取得・予想・保存（土曜夜のみ） ──

jst_now = datetime.now(JST)
next_date = (jst_now + timedelta(days=1)).strftime('%Y%m%d')
print(f'📅 取得日: {next_date}（日曜）')

races = fetch_races_on_date(sess, next_date, HIST_PATH)
print(f'📋 取得レース: {len(races)}R')

surf_counts = {}
for r in races:
    s = r.get('surface', '?')
    surf_counts[s] = surf_counts.get(s, 0) + 1
print(f'   コース内訳: {surf_counts}')

selected = ability_first_loose(races, avg_bias, top_n=TOP_N_RACES)
print(f'⭐ 厳選: {len(selected)}レース\n')

total_inv = 0
for i, c in enumerate(selected, 1):
    race = c['race']; top1 = c['top1']; scored = c['scored']
    bets = make_bets(c)
    invest = sum(b['amount'] for b in bets)
    total_inv += invest
    print(f'【{i}】{race["racecourse"]} R{race["race_num"]:02d} {race.get("race_name","")}'
          f'  {race["distance"]}m{race["surface"]}  {race.get("num_horses",0)}頭')
    print(f'  ◎ #{top1["num"]} {top1["name"]}  {top1.get("win_odds",0):.1f}倍  スコア:{top1["total"]:.2f}')
    for b in bets:
        print(f'  {b["type"]} #{b["nums"][0]} ¥{b["amount"]:,}  EV:{b["ev"]:.2f}')
    print(f'  投資: ¥{invest:,}\n')
    save_race_db(race, BASE_DIR)
    save_bets_db(race['date'], race['id'], bets, BASE_DIR)
    log_bet_simulation(race['date'], c, BASE_DIR)

print(f'💰 投資合計: ¥{total_inv:,}')

jst_now = datetime.now(JST)
app_data = to_app_json(selected, races, avg_bias, jst_now, day_type='sunday')
app_path = f'{BASE_DIR}/app/latest.json'
with open(app_path, 'w', encoding='utf-8') as f:
    json.dump(app_data, f, ensure_ascii=False, indent=2)
print(f'✅ アプリJSON保存: {app_path}')

In [ ]:
# ── セル7: GitHub Pagesにプッシュ（土曜夜のみ） ──

import base64, json as _json

def push_to_github(data, pat):
    url = 'https://api.github.com/repos/hanagenuku/keiba_ai/contents/data/latest.json'
    headers = {'Authorization': f'token {pat}', 'Accept': 'application/vnd.github.v3+json'}
    sha = requests.get(url, headers=headers).json().get('sha')
    content = base64.b64encode(_json.dumps(data, ensure_ascii=False, indent=2).encode()).decode()
    payload = {'message': 'update predictions', 'content': content, 'sha': sha}
    r = requests.put(url, headers=headers, json=payload)
    print('✅ GitHub Pagesにプッシュ完了' if r.status_code in (200, 201) else f'⚠️ 失敗: {r.status_code}')

from google.colab import userdata
GITHUB_PAT = userdata.get('GITHUB_PAT')
push_to_github(app_data, GITHUB_PAT)